In [12]:
import pandas as pd

## Severity Tier Construction

We construct severity tiers using final_severity_level_code, which represents the final assessed urgency of the incident. Lower numeric codes correspond to higher urgency. We group codes as:

High: 1–2

Medium: 3–4

Low: 5–8

We also create severity_tier_numeric (High=1, Medium=2, Low=3) and high_severity (1 if High else 0) for downstream regression and conditional comparisons.

In [13]:
# load merged EMS + ACS dataset
merged = pd.read_parquet("../data/processed/merged_ems_acs_w_groups.parquet")
merged.head()
merged.columns

Index(['cad_incident_id', 'incident_datetime', 'initial_call_type',
       'initial_severity_level_code', 'final_call_type',
       'final_severity_level_code', 'valid_dispatch_rspns_time_indc',
       'dispatch_response_seconds_qy', 'valid_incident_rspns_time_indc',
       'incident_response_seconds_qy', 'incident_travel_tm_seconds_qy',
       'held_indicator', 'incident_disposition_code', 'borough', 'zipcode',
       'median_income', 'total_pop', 'white_pop', 'black_pop', 'hispanic_pop',
       'pct_black', 'pct_white', 'pct_hispanic', 'income_q', 'high_black_zip'],
      dtype='object')

In [14]:
# check missingness in severity codes
print("Missing rate:", merged['final_severity_level_code'].isna().mean())

# inspect severity code distribution
merged['final_severity_level_code'].value_counts().sort_index()

Missing rate: 0.0


final_severity_level_code
1     6933
2    79637
3    58792
4    65857
5    61534
6    54746
7    35574
8      410
Name: count, dtype: int64

In [15]:
# map final severity codes into 3-tier severity groups
def map_severity(code):
    if pd.isna(code):
        return None
    
    code = int(code)
    
    if code in (1, 2):
        return "High"
    elif code in (3, 4):
        return "Medium"
    elif code in (5, 6, 7, 8):
        return "Low"
    else:
        return None

merged['severity_tier'] = merged['final_severity_level_code'].apply(map_severity)

In [16]:
# summarize tier distribution
counts = merged["severity_tier"].value_counts(dropna=False)
props = merged["severity_tier"].value_counts(normalize=True, dropna=False)

severity_summary = pd.DataFrame({
    "count": counts,
    "proportion": props
})

severity_summary

,count,proportion
severity_tier,,
Low,152264,0.418903
Medium,124649,0.342929
High,86570,0.238168


In [17]:
# create numeric severity encoding for regression use
severity_map_numeric = {
    "High": 1,
    "Medium": 2,
    "Low": 3
}

merged["severity_tier_numeric"] = merged["severity_tier"].map(severity_map_numeric)

merged["high_severity"] = (merged["severity_tier"] == "High").astype(int)

merged.groupby("severity_tier")["incident_response_seconds_qy"].mean().sort_index()

severity_tier
High      462.367691
Low       894.969448
Medium    616.125769
Name: incident_response_seconds_qy, dtype: float64

In [18]:
merged[['final_severity_level_code','final_call_type']].head(10)

,final_severity_level_code,final_call_type
0,5,INJURY
1,3,CARD
2,4,RESPIR
3,4,BURNMI
4,7,EDPC
5,5,INJURY
6,5,INJURY
7,2,UNC
8,5,ABDPN
9,2,ASTHMB


In [19]:
# top call types within each severity tier
(
    merged.groupby("severity_tier")["final_call_type"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)

severity_tier  final_call_type
High           CARDBR             26608
               UNC                18048
               DIFFBR             16611
               STATEP              6520
               ARREST              6452
Low            SICK               50176
               INJURY             38982
               ABDPN              17428
               EDPC               14580
               EDP                 9231
Medium         UNKNOW             26655
               DRUG               21632
               CARD               20246
               INJMAJ             14570
               MVAINJ              9030
Name: count, dtype: int64

In [20]:
# save dataset with severity tiers to data/processed
severity_out = "../data/processed/merged_ems_acs_with_severity.parquet"
merged.to_parquet(severity_out, index=False)
print("saved:", severity_out)

saved: ../data/processed/merged_ems_acs_with_severity.parquet
